In [10]:
# Load env variables and create client
import os
import debugpy
import json
from dotenv import load_dotenv
from anthropic import Anthropic
from IPython.display import display
import pprint, shutil
import textwrap

load_dotenv()
if not os.getenv("ANTHROPIC_API_KEY"):
    exit("No API key found in environment variables")

client = Anthropic()
model = "claude-haiku-4-5"

def myprint(messages):
    width = shutil.get_terminal_size().columns
    for line in json.dumps(messages, indent=2, ensure_ascii=False).splitlines():
        indent = len(line) - len(line.lstrip())
        print(textwrap.fill(line, width=width, subsequent_indent=' ' * (indent + 2)))

def myprint2(answer):
    width = shutil.get_terminal_size().columns
    for line in answer.splitlines():
        indent = len(line) - len(line.lstrip())
        print(textwrap.fill(line, width=width, subsequent_indent=' ' * (indent + 2)))

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [16]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
print(text)


{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}



In [18]:
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())
print(json.dumps(clean_json, indent=4))

{
    "Name": "MySimpleRule",
    "EventBusName": "default",
    "EventPattern": {
        "source": [
            "aws.ec2"
        ],
        "detail-type": [
            "EC2 Instance State-change Notification"
        ],
        "detail": {
            "state": [
                "running"
            ]
        }
    },
    "State": "ENABLED",
    "Targets": [
        {
            "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
            "Id": "1"
        }
    ]
}
